# In-context forecasting: a hands-on walkthrough

TiRex-2 is a pretrained time-series foundation model: it forecasts a series **zero-shot**, using only the
history you hand it as *context*. This notebook shows, on a real hydraulic test rig, how much that context matters.
Everything uses a single pair of cycles: cycle **1787** is the context, cycle **1788** is the ground truth to forecast.

1. one cycle of context is already enough for a first forecast,
2. more context (the same cycle repeated 2-5 times) makes the forecast sharper,
3. a **covariate** (a second, related sensor) can add further information on top.

Every figure is interactive: hover for exact values, click legend entries to hide/show traces, drag the slider to change the context length.

**Glossary**

| Term | Meaning |
|---|---|
| **Cycle** | One 60 s run of the rig's load profile. Each sensor is stored as 200 points per cycle (0.3 s resolution). |
| **PS1 / EPS1** | PS1 = pressure sensor (bar). EPS1 = motor power (W). |
| **Context** | The history given to the model. Here: cycle 1787, optionally repeated *n* times back to back. |
| **Forecast horizon** | What the model predicts: the next full cycle (200 points = 60 s), compared with the real cycle 1788. |
| **10-90 % band** | The range between the model's 10 % and 90 % quantile forecasts: the model expects the truth to fall inside it 80 % of the time. |
| **Covariate** | An extra series (here EPS1) given alongside the target (PS1), so the model can use it as additional context. |
| **MAE** | Mean absolute error `mean(|forecast - truth|)`, in bar. Lower is better. |

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
from plotly.colors import hex_to_rgb
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model
from tirex2.plotting import ALPHA_QUANTILES, COLOR_CONTEXT, COLOR_CUTOFF_LINE, COLOR_FORECAST, COLOR_GROUND_TRUTH, COVARIATE_COLORS

pio.renderers.default = "plotly_mimetype+notebook_connected"  # works in Jupyter, VS Code and HTML export

CYCLE_S = 60                # seconds per cycle
POINTS = 200                # samples per cycle
N_LIST = (1, 2, 3, 4, 5)    # context lengths, in repeated cycles
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

# colour palette: TiRex-2's own plotting colours (tirex2.plotting), so the figures match the docs site
C_CTX, C_TRUTH = COLOR_CONTEXT, COLOR_GROUND_TRUTH   # PS1 context and ground truth (blue)
C_UNI, C_COV = COLOR_FORECAST, COVARIATE_COLORS[1]  # univariate forecast (red), +EPS1 forecast (green)
C_EPS1 = COVARIATE_COLORS[0]                        # the EPS1 signal itself (orange)
FONT = "Roboto, -apple-system, 'Segoe UI', Helvetica, Arial, sans-serif"  # Material for MkDocs default font

C_TEXT, C_MUTED, C_GRID = "#1e293b", "#64748b", "#eef2f7"

pio.templates["doc"] = go.layout.Template(pio.templates["plotly_white"])
pio.templates["doc"].layout.update(
    font=dict(family=FONT, size=13, color=C_TEXT),
    title=dict(x=0.0, xanchor="left", y=0.98, yanchor="top", yref="container",
               font=dict(size=19, color=C_TEXT, weight=700), subtitle=dict(font=dict(size=13, color=C_MUTED))),
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", bordercolor="#e2e8f0", font=dict(family=FONT, size=12, color=C_TEXT)),
    legend=dict(orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5, bgcolor="rgba(0,0,0,0)", font=dict(size=12, color=C_MUTED)),
    margin=dict(l=70, r=30, t=150, b=100),
    xaxis=dict(showline=False, zeroline=False, ticks="", gridcolor=C_GRID, tickfont=dict(size=12, color=C_MUTED),
               showspikes=True, spikemode="across", spikethickness=1, spikecolor="#cbd5e1", spikedash="dot", spikesnap="cursor"),
    yaxis=dict(showline=False, zeroline=False, ticks="", gridcolor=C_GRID, tickfont=dict(size=12, color=C_MUTED)),
)
pio.templates.default = "doc"

# interactive-toolbar config shared by every figure: no logo, only shown on hover, no selection tools
CONFIG = dict(displaylogo=False, displayModeBar="hover", modeBarButtonsToRemove=["lasso2d", "select2d", "autoScale2d"])


# Set to a folder (e.g. Path("docs_export")) to also write every figure as a standalone HTML file for the docs site.
EXPORT_DIR = None


def show(fig, name: str):
    fig.show(config=CONFIG)
    if EXPORT_DIR is not None:
        Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
        fig.write_html(Path(EXPORT_DIR) / f"{name}.html", include_plotlyjs="cdn", full_html=True, config=CONFIG, auto_play=False,
                       default_width="100%", default_height=f"{fig.layout.height}px")

/Users/maria.zuliani/Projects/tirex2_in-context/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the dataset

`dataset.txt` holds 10 consecutive, healthy, stable cycles of two sensors (PS1 and EPS1), already block-averaged
to 200 points per cycle. Each row is one `(sensor, cycle)` pair. We use only two of them: **1787** (context) and **1788** (target).

In [2]:
DATA_PATH = next(p for p in [Path("dataset.txt"), Path("final_documentation/dataset.txt")] if p.exists())
df = pd.read_csv(DATA_PATH, sep="\t")

CYCLE_IDS = sorted(df["cycle"].unique())
data = {s: g.sort_values("cycle").drop(columns=["sensor", "cycle"]).to_numpy(float) for s, g in df.groupby("sensor")}
t_cycle = np.arange(POINTS) * CYCLE_S / POINTS   # seconds within a cycle

# The whole notebook uses one pair: cycle 1787 is the context, cycle 1788 is the ground truth to forecast
CONTEXT_ID, TARGET_ID = 1787, 1788
CTX_POS, TGT_POS = CYCLE_IDS.index(CONTEXT_ID), CYCLE_IDS.index(TARGET_ID)
print(f"{len(CYCLE_IDS)} cycles ({CYCLE_IDS[0]}-{CYCLE_IDS[-1]}), sensors: {list(data)}, shape per sensor: {data['PS1'].shape}")
print(f"context cycle: {CONTEXT_ID}, target (ground truth) cycle: {TARGET_ID}")
df.iloc[:, :8]

10 cycles (1787-1796), sensors: ['EPS1', 'PS1'], shape per sensor: (10, 200)
context cycle: 1787, target (ground truth) cycle: 1788


,sensor,cycle,0,1,2,3,4,5
0,PS1,1787,156.663333,172.503333,188.023000,191.340000,191.481333,191.444000
1,PS1,1788,156.637333,172.443333,188.007667,191.392333,191.444667,191.465333
2,PS1,1789,156.610667,172.505000,187.998000,191.401333,191.494667,191.478000
3,PS1,1790,156.617000,172.483333,188.012333,191.396667,191.497667,191.432000
4,PS1,1791,156.543667,172.373000,187.967333,191.335000,191.463333,191.435333
5,PS1,1792,156.590000,172.466000,188.017667,191.410000,191.465333,191.389667
6,PS1,1793,156.526000,172.344333,187.989000,191.405000,191.454333,191.422000
7,PS1,1794,156.534000,172.400333,187.997667,191.438333,191.508667,191.410000
8,PS1,1795,156.648000,172.477667,187.999333,191.394333,191.486333,191.443667
9,PS1,1796,156.585000,172.354667,187.960000,191.405667,191.443333,191.356333


## 2. The context cycle and the target cycle

A cycle is a fixed 60 s load profile, so consecutive cycles are almost identical. The plot overlays the context
cycle 1787 (solid) and the target cycle 1788 (dotted) for both sensors. PS1 (pressure) and EPS1 (motor power) live on
very different scales, but they move together.

In [3]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("PS1 · pressure (bar)", "EPS1 · motor power (W)"))
for sensor, color, unit, fmt, row in [("PS1", C_CTX, "bar", ".1f", 1), ("EPS1", C_EPS1, "W", ".0f", 2)]:
    for pos, role, dash in [(CTX_POS, "context", "solid"), (TGT_POS, "target", "dot")]:
        label = f"{sensor} · cycle {CYCLE_IDS[pos]} ({role})"
        fig.add_trace(go.Scatter(x=t_cycle, y=data[sensor][pos], name=label, mode="lines", line=dict(color=color, width=3, dash=dash),
                                 hovertemplate=f"%{{y:{fmt}}} {unit}<extra>cycle {CYCLE_IDS[pos]}</extra>"), row=row, col=1)

fig.update_layout(
    title=dict(text=f"Context cycle {CONTEXT_ID} vs. target cycle {TARGET_ID}",
               subtitle=dict(text="solid = context given to the model  ·  dotted = the cycle it has to forecast")),
    height=640, margin=dict(t=120), legend_y=-0.12,
)
fig.update_xaxes(title_text="time in cycle (s)", row=2, col=1)
fig.update_yaxes(title_text="bar", row=1, col=1)
fig.update_yaxes(title_text="W", row=2, col=1)
show(fig, "context-vs-target")

## 3. Forecasting cycle 1788 from one cycle of context

We load TiRex-2 and give it **one** cycle of PS1: cycle 1787. It forecasts the next 200 points as a set of quantiles.
We plot the context, the median forecast with its 10-90 % band, and the real cycle 1788 (ground truth).
The subtitle reports the MAE.

In [4]:
model = load_model("NX-AI/TiRex-2", device=DEVICE)
Q = model.quantiles.detach().cpu().numpy()
MED, LO, HI = int(np.argmin(abs(Q - 0.5))), int(np.argmin(abs(Q - 0.1))), int(np.argmin(abs(Q - 0.9)))
print(f"TiRex-2 on {DEVICE}; quantiles {Q.round(2).tolist()} -> using p10={Q[LO]:.1f}, median={Q[MED]:.1f}, p90={Q[HI]:.1f}")


def forecast(context: np.ndarray, covariate: np.ndarray | None = None):
    # Forecast the next POINTS values; returns (median, p10, p90).
    target = torch.tensor(context, dtype=torch.float32).unsqueeze(0)                                    # [1, T]
    past_cov = torch.tensor(covariate, dtype=torch.float32).unsqueeze(0) if covariate is not None else None  # [1, T]
    ts = TimeseriesType(target=target, past_covariates=past_cov, future_covariates=None)
    fc = model.forecast([ts], prediction_length=POINTS, output_type="numpy")[0]
    return fc[0, MED], fc[0, LO], fc[0, HI]


def mae(pred, truth):
    return float(np.mean(np.abs(pred - truth)))


_cache = {}


def run(n: int, covariate: bool = False) -> dict:
    # Forecast the target cycle of PS1 from the context cycle repeated n times (optionally with EPS1 as covariate).
    key = (n, covariate)
    if key not in _cache:
        ctx = np.tile(data["PS1"][CTX_POS], n)
        cov = np.tile(data["EPS1"][CTX_POS], n) if covariate else None
        pred, lo, hi = forecast(ctx, cov)
        _cache[key] = dict(pred=pred, lo=lo, hi=hi, mae=mae(pred, data["PS1"][TGT_POS]))
    return _cache[key]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1845.27it/s]

TiRex-2 on mps; quantiles [0.10000000149011612, 0.20000000298023224, 0.30000001192092896, 0.4000000059604645, 0.5, 0.6000000238418579, 0.699999988079071, 0.800000011920929, 0.8999999761581421] -> using p10=0.1, median=0.5, p90=0.9


In [5]:
def _segments(cycle: np.ndarray, n: int):
    # n repeated copies of `cycle` ending at t=0, separated by NaN gaps so the cycle boundaries are visible.
    xs, ys = [], []
    for j in range(n):
        xs += [*(t_cycle + (j - n) * CYCLE_S), np.nan]
        ys += [*cycle, np.nan]
    return np.array(xs), np.array(ys)


def _rgba(hex_color: str, alpha: float) -> str:
    return "rgba({},{},{},{})".format(*hex_to_rgb(hex_color), alpha)


def _title(n: int, covariate: bool) -> dict:
    # Title + subtitle dict, so frames can swap it as the context length changes.
    u = run(n, False)["mae"]
    plural = "cycle" if n == 1 else "cycles"
    pair = f"context {CONTEXT_ID} → target {TARGET_ID}"
    if not covariate:
        return dict(text=f"PS1 forecast · context = {n} {plural}", subtitle=dict(text=f"<b>MAE = {u:.2f} bar</b>  ·  {pair}"))
    c = run(n, True)["mae"]
    change = (c - u) / u * 100 + 0.0
    change = 0.0 if abs(change) < 0.5 else change
    return dict(text=f"PS1 forecast · context = {n} {plural}",
                subtitle=dict(text=f"MAE: univariate <b>{u:.2f} bar</b>  →  +EPS1 covariate <b>{c:.2f} bar</b>  (error {change:+.0f} %)  ·  {pair}"))


def _traces(n: int, covariate: bool) -> list:
    truth = data["PS1"][TGT_POS]
    cx, cy = _segments(data["PS1"][CTX_POS], n)
    out = [
        go.Scatter(x=cx, y=cy, name="context (PS1)", mode="lines", line=dict(color=C_CTX, width=2.5),
                   hovertemplate="%{y:.1f} bar<extra>context</extra>", xaxis="x", yaxis="y"),
        go.Scatter(x=t_cycle, y=truth, name="ground truth", mode="lines", line=dict(color=C_TRUTH, width=3, dash="dot"),
                   hovertemplate="%{y:.1f} bar<extra>ground truth</extra>", xaxis="x", yaxis="y"),
    ]

    def forecast_group(res, color, label, group):
        return [
            go.Scatter(x=t_cycle, y=res["lo"], mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False,
                       legendgroup=group, xaxis="x", yaxis="y"),
            go.Scatter(x=t_cycle, y=res["hi"], mode="lines", line=dict(width=0), fill="tonexty", fillcolor=_rgba(color, 2 * ALPHA_QUANTILES),
                       hoverinfo="skip", showlegend=False, legendgroup=group, xaxis="x", yaxis="y"),
            go.Scatter(x=t_cycle, y=res["pred"], name=label, mode="lines", line=dict(color=color, width=3, dash="solid"),
                       customdata=np.column_stack([res["lo"], res["hi"]]), legendgroup=group,
                       hovertemplate="%{y:.1f} bar  (10-90 %: %{customdata[0]:.1f} – %{customdata[1]:.1f})<extra>" + label + "</extra>",
                       xaxis="x", yaxis="y"),
        ]

    uni_label = "univariate forecast (median + 10-90 % band)" if covariate else "TiRex-2 forecast (median + 10-90 % band)"
    out += forecast_group(run(n, False), C_UNI, uni_label, "uni")
    if covariate:
        out += forecast_group(run(n, True), C_COV, "+EPS1 covariate forecast (median + 10-90 % band)", "cov")
        ex, ey = _segments(data["EPS1"][CTX_POS], n)
        out.append(go.Scatter(x=ex, y=ey, name="EPS1 covariate context", mode="lines", line=dict(color=C_EPS1, width=2.5),
                              hovertemplate="%{y:.0f} W<extra>EPS1 context</extra>", xaxis="x2", yaxis="y2"))
    return out


def forecast_figure(n_list, covariate: bool = False) -> go.Figure:
    # Forecast vs. ground truth. With several n, adds a slider (and play button) over the context length.
    rows = 2 if covariate else 1
    fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        row_heights=[0.74, 0.26] if covariate else None)
    frames = [go.Frame(data=_traces(n, covariate), name=str(n), layout=go.Layout(title=_title(n, covariate)))
              for n in n_list]
    fig.add_traces(frames[0].data)

    # fixed axes so changing n only changes what is drawn, not the scale
    ps1_all = np.concatenate([data["PS1"][TGT_POS]] + [run(n, c)[k] for n in n_list for c in ([False, True] if covariate else [False]) for k in ("lo", "hi")])
    pad = 0.05 * np.ptp(ps1_all)
    fig.update_yaxes(title_text="PS1 (bar)", range=[ps1_all.min() - pad, ps1_all.max() + pad], row=1, col=1)
    if covariate:
        e = data["EPS1"][CTX_POS]
        fig.update_yaxes(title_text="EPS1 (W)", range=[e.min() - 0.08 * np.ptp(e), e.max() + 0.08 * np.ptp(e)], row=2, col=1)
    max_n = max(n_list)
    fig.update_xaxes(range=[-max_n * CYCLE_S - 3, CYCLE_S + 3], title_text="time relative to forecast start (s)", row=rows, col=1)

    fig.add_vrect(x0=0, x1=CYCLE_S, fillcolor=_rgba(C_UNI, 0.03), line_width=0, layer="below", row="all", col="all")
    fig.add_vline(x=0, line=dict(color=COLOR_CUTOFF_LINE, width=1.2, dash="dot"), row="all", col="all")
    for x, text in [(CYCLE_S / 2, "forecast"), (-CYCLE_S / 2, "context")]:
        fig.add_annotation(x=x, y=1, xref="x", yref="y domain", text=text.upper(), showarrow=False,
                           yanchor="top", font=dict(color="#94a3b8", size=11))

    fig.update_layout(title=_title(n_list[0], covariate), height=780 if covariate else 580,
                      legend_y=-0.2)
    if len(n_list) > 1:
        fig.frames = frames
        no_anim = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
        fig.update_layout(
            # legend on top, so the bottom is free for the context-length slider and the play button
            legend=dict(yanchor="bottom", y=1.02, xanchor="left", x=0.0),
            margin=dict(t=170 if covariate else 150, b=140),
            sliders=[dict(active=0, x=0.14, len=0.84, pad=dict(t=50, b=0), ticklen=0, minorticklen=0,
                          bgcolor="#e2e8f0", activebgcolor=C_TEXT, bordercolor="#e2e8f0", tickcolor="#94a3b8",
                          font=dict(size=12, color=C_MUTED),
                          currentvalue=dict(prefix="Context length: ", font=dict(size=14, color=C_TEXT)),
                          steps=[dict(method="animate", args=[[str(n)], no_anim], label=f"{n} cycle{'s' if n > 1 else ''}") for n in n_list])],
            updatemenus=[dict(type="buttons", direction="left", x=0.0, y=-0.02, xanchor="left", yanchor="top", pad=dict(t=50),
                              bgcolor="white", bordercolor="#cbd5e1", borderwidth=1, font=dict(size=13, color=C_TEXT), showactive=False,
                              buttons=[dict(label="▶  Play", method="animate",
                                            args=[None, dict(frame=dict(duration=1300, redraw=True), fromcurrent=True, transition=dict(duration=0))])])],
        )
    else:
        fig.update_layout(margin=dict(t=100))
    return fig


show(forecast_figure([1]), "forecast-one-cycle")

## 4. More context: repeating cycle 1787 2, 3, 4, 5 times

Same forecast, same target cycle 1788, but now the context is cycle 1787 **repeated and concatenated** *n* times
(`n = 1 ... 5`). No new information about the process is added, only repetition, which tells the model the pattern is
periodic and how stable it is. Drag the slider (or press play) to watch the forecast and its MAE change with *n*.

In [6]:
show(forecast_figure(N_LIST), "forecast-context-slider")

### Does the error drop as context grows?

The MAE of the forecast of cycle 1788 for each context length. The label on the right is the change from 1 to 5 cycles.

In [7]:
VARIANTS = {"univariate": (False, C_UNI), "+EPS1 covariate": (True, C_COV)}


def metric_figure(variant_names, title: str, subtitle: str) -> go.Figure:
    # MAE of the target-cycle forecast for each context length, one line per variant.
    fig = go.Figure()
    for i, name in enumerate(variant_names):
        cov, color = VARIANTS[name]
        err = [run(n, cov)["mae"] for n in N_LIST]
        fig.add_trace(go.Scatter(x=list(N_LIST), y=err, name=name, mode="lines+markers",
                                 line=dict(color=color, width=4), marker=dict(size=11, color="white", line=dict(color=color, width=3)),
                                 hovertemplate=f"{name}: " + "%{y:.2f} bar<extra></extra>"))
        drop = (err[-1] - err[0]) / err[0] * 100
        fig.add_annotation(x=N_LIST[-1], y=err[-1], text=f"<b>{drop:+.0f} %</b>", showarrow=False, xanchor="left", xshift=14,
                           yshift=11 * (-1) ** i if len(variant_names) > 1 else 0, font=dict(color=color, size=14))
    fig.update_layout(title=dict(text=title, subtitle=dict(text=subtitle)), height=500, margin=dict(t=110), legend_y=-0.22)
    fig.update_xaxes(title_text="context length (repeated cycles)", tickmode="array", tickvals=list(N_LIST),
                     ticktext=[f"{n} cycle{'s' if n > 1 else ''}" for n in N_LIST], range=[0.8, N_LIST[-1] + 0.5])
    fig.update_yaxes(title_text="MAE (bar, lower is better)", rangemode="tozero")
    return fig


show(metric_figure(["univariate"], "Forecast error vs. context length",
                   f"PS1, univariate  ·  context {CONTEXT_ID} → target {TARGET_ID}  ·  % = change from 1 to 5 cycles"),
     "error-vs-context")

## 5. Univariate vs. using EPS1 as a covariate

TiRex-2 can also look at *other* series while forecasting the target. Here we pass EPS1 (motor power) of cycle 1787 as
`past_covariates`, with the same repetition as PS1 (the covariate context must have the same length as the target context).
The green forecast uses PS1 **and** EPS1; the red one uses PS1 only. The lower panel shows exactly which extra signal the model sees.

In [8]:
show(forecast_figure(N_LIST, covariate=True), "forecast-covariate-slider")

### Does the covariate help at every context length?

In [9]:
show(metric_figure(list(VARIANTS), "Forecast error: univariate vs. +EPS1",
                   f"context {CONTEXT_ID} → target {TARGET_ID}  ·  % = change from 1 to 5 cycles"),
     "error-univariate-vs-covariate")

table = pd.DataFrame({name: {n: run(n, cov)["mae"] for n in N_LIST} for name, (cov, _) in VARIANTS.items()}).T
table.loc["covariate gain (%)"] = (table.loc["univariate"] - table.loc["+EPS1 covariate"]) / table.loc["univariate"] * 100
table.columns = [f"{n} cycle{'s' if n > 1 else ''}" for n in N_LIST]
print(f"MAE (bar), context cycle {CONTEXT_ID} -> target cycle {TARGET_ID}")
if EXPORT_DIR is not None:  # the same table as Markdown, ready to paste into the docs page
    md = table.map(lambda v: f"{v:.2f}").reset_index(names="")
    lines = ["| " + " | ".join(md.columns) + " |", "|" + " :--- |" + " ---: |" * (md.shape[1] - 1)]
    lines += ["| " + " | ".join(row) + " |" for row in md.to_numpy()]
    (Path(EXPORT_DIR) / "results-table.md").write_text("\n".join(lines) + "\n")
table.round(2)

MAE (bar), context cycle 1787 -> target cycle 1788


,1 cycle,2 cycles,3 cycles,4 cycles,5 cycles
univariate,11.02,5.58,2.45,1.64,1.29
+EPS1 covariate,11.01,3.26,1.99,1.43,1.23
covariate gain (%),0.13,41.63,19.02,13.21,5.16


## Take away

MAE of the forecast of cycle 1788 from cycle 1787 (lower is better).

| MAE (bar) | 1 cycle | 2 cycles | 3 cycles | 4 cycles | 5 cycles |
| :--- | ---: | ---: | ---: | ---: | ---: |
| univariate | 11.02 | 5.58 | 2.45 | 1.64 | 1.29 |
| +EPS1 covariate | 11.01 | 3.26 | 1.99 | 1.43 | 1.23 |
| covariate gain | 0 % | 42 % | 19 % | 13 % | 5 % |

- **One cycle is not enough to see the pattern.** With a single cycle of context the forecast is flat and cautious
  (MAE ≈ 11.0 bar): the model has seen the shape once, but cannot know that the cycle repeats.
- **Repeating the same cycle is the cheapest way to improve the forecast.** The univariate error falls from 11.02 bar
  (1 cycle) to 5.58 (2), 2.45 (3), 1.64 (4) and 1.29 bar (5), about an 88 % reduction. No new information is added, only the
  evidence that the pattern is periodic. The improvement gets smaller after ~3-4 cycles.
- **A covariate helps most when the context is short, but still too short to be self-explanatory.** With 1 cycle EPS1 changes
  nothing (11.02 vs 11.01 bar); at 2 cycles it cuts the error by 42 % (5.58 → 3.26 bar); the gain then shrinks (19 %, 13 %, 5 %) as the target's
  own history becomes sufficient.
- **Practical reading:** if you can only provide 2-3 cycles of history, add related sensors as covariates; with 5 or more cycles the univariate
  forecast is already close to the covariate one.

*Caveat: a single context/target pair (1787 → 1788) from one healthy operating condition. This is one example, not a statistical result.*